# Variational ECD+R decomposition of the two-mode squeezing gate

This notebook searches for the parameters of an **ECD + qubit-rotation (ECD+R)** circuit
that approximates the two-mode squeezing (TMS) unitary

$$
S_{AB}(r) \;=\; \exp\!\bigl[\, r\,(a_A^{\dagger} a_B^{\dagger} - a_A a_B)\,\bigr]
$$

acting on two bosonic modes $A$ and $B$ with the help of a single dispersively
coupled two-level ancilla ("qubit"). This is the standard hybrid CV–DV setting
used in circuit-QED experiments, where the qubit mediates non-Gaussian control
over otherwise-linear cavities.

The variational ansatz is built from three primitives:

1. **Echoed Conditional Displacement on mode $A$**,
   $\mathrm{ECD}_A(\beta) = |g\rangle\langle g|\otimes D_A(\beta/2)\otimes I_B
                          + |e\rangle\langle e|\otimes D_A(-\beta/2)\otimes I_B$.
2. **Echoed Conditional Displacement on mode $B$**,
   $\mathrm{ECD}_B(\beta) = |g\rangle\langle g|\otimes I_A\otimes D_B(\beta/2)
                          + |e\rangle\langle e|\otimes I_A\otimes D_B(-\beta/2)$.
3. A **qubit rotation** $R(\theta,\varphi) = \exp\!\bigl[-\tfrac{i\theta}{2}
   (\cos\varphi\,X + \sin\varphi\,Y)\bigr]$.

A layer consists of $\mathrm{ECD}_A \to \mathrm{ECD}_B \to R$, and the ansatz
stacks $N_{\text{layers}}$ of them. After the sequence, the qubit is
post-selected on $|g\rangle$; the remaining two-mode state is compared to the
target state $S_{AB}(r)|0,0\rangle$ (a two-mode squeezed vacuum) via
fidelity.


## ECD+R circuit diagram

The variational circuit repeats the block below $N_{\text{layers}}$ times and
finally projects the ancilla onto $|g\rangle$:

![ECD+R sequence](ecd_r_sequence.png)

The vertical lines between the ancilla wire and the cavity gates indicate the
qubit-conditioned nature of the $\mathrm{ECD}$ operation: the direction of the
displacement in phase space is flipped depending on the qubit state
$\{|g\rangle,|e\rangle\}$. Each layer therefore entangles the qubit with the
cavities and, together with the subsequent qubit rotation, can imprint
non-Gaussian correlations onto the joint two-mode state.


## 1. Imports


In [ ]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize


## 2. Hilbert-space setup

We truncate each bosonic mode at a finite Fock cutoff. Mode $A$ carries most of
the squeezing amplitude and needs a larger cutoff; mode $B$ can be kept small
for small target squeezing $r$, but should be increased whenever
$\sinh^{2}(r)$ approaches the truncation.

The ordering of the tensor product throughout the notebook is
$\mathcal{H} = \mathcal{H}_{\text{qubit}} \otimes \mathcal{H}_A \otimes \mathcal{H}_B$,
so a global operator built as `np.kron(Q, np.kron(OA, OB))` acts as $Q$ on the
qubit, $O_A$ on mode $A$, and $O_B$ on mode $B$.


In [ ]:
# ---- Truncation dimensions ------------------------------------------------
cutoffA = 10     # Fock cutoff of mode A (carries the bulk of the squeezing)
cutoffB = 10     # Fock cutoff of mode B (must be large enough for sinh^2(r))

# ---- Variational-circuit hyperparameters ---------------------------------
Nlayers  = 1     # number of ECD_A -> ECD_B -> R layers in the ansatz
r_target = 0.5   # target two-mode squeezing parameter


In [ ]:
def destroy(dim):
    """Truncated bosonic annihilation operator on a Fock space of size `dim`.

    Matrix elements: a|n> = sqrt(n) |n-1>. The truncation drops the coupling
    from |dim> to |dim-1>, which is fine as long as the target state has
    negligible support on |dim>.
    """
    a = np.zeros((dim, dim), dtype=complex)
    for n in range(1, dim):
        a[n - 1, n] = np.sqrt(n)
    return a


# Mode operators and identities on each cavity subspace
aA, IA = destroy(cutoffA), np.eye(cutoffA)
adagA  = aA.conj().T

aB, IB = destroy(cutoffB), np.eye(cutoffB)
adagB  = aB.conj().T


# ---- Qubit (ancilla) operators ------------------------------------------
# Pauli X, Y used inside the rotation generator
X = np.array([[0, 1],  [1, 0]],   dtype=complex)
Y = np.array([[0, -1j],[1j, 0]],  dtype=complex)

# Projectors onto |g> and |e> used by the ECD conditional structure
Pg = np.array([[1, 0], [0, 0]],   dtype=complex)
Pe = np.array([[0, 0], [0, 1]],   dtype=complex)


## 3. Circuit primitives

Three primitives are built here:

* `displacement(alpha, mode)` — the usual bosonic displacement
  $D(\alpha) = \exp(\alpha a^\dagger - \alpha^{*} a)$ on either cavity.
* `Rphi(theta, phi)` — a general single-qubit rotation about an axis in the
  equatorial plane, $R(\theta,\varphi) = \exp[-i\theta(\cos\varphi\,X + \sin\varphi\,Y)/2]$.
* `ECD_A`, `ECD_B` — the qubit-conditioned "echoed" displacements. In the
  ideal (fast-gate) limit these are the workhorse of the ECD control scheme:
  they entangle the qubit's Pauli-$Z$ eigenstates with equal-and-opposite
  displacements of the cavity, allowing an interleaved qubit rotation to steer
  the cavity through a rich non-Gaussian trajectory.

Note that we compute displacements by matrix exponentiation of the *truncated*
$\alpha a^\dagger - \alpha^{*} a$. This is unitary only within the truncated
subspace and is accurate whenever the cavity population stays well below the
cutoff.


In [ ]:
def displacement(alpha, mode):
    """Bosonic displacement operator D(alpha) on the requested cavity mode."""
    if mode == "A":
        return expm(alpha * adagA - np.conjugate(alpha) * aA)
    elif mode == "B":
        return expm(alpha * adagB - np.conjugate(alpha) * aB)
    else:
        raise ValueError("mode must be 'A' or 'B'")


def Rphi(theta, phi):
    """Qubit rotation by angle `theta` about axis (cos phi, sin phi, 0)."""
    sigma = np.cos(phi) * X + np.sin(phi) * Y
    return expm(-1j * theta * sigma / 2)


def ECD_A(beta):
    """Echoed Conditional Displacement on mode A, conditioned on the qubit.

    ECD_A(beta) = |g><g| (x) D_A(+beta/2) (x) I_B
                + |e><e| (x) D_A(-beta/2) (x) I_B
    """
    Dp = displacement( beta / 2, "A")
    Dm = displacement(-beta / 2, "A")
    return (
        np.kron(Pg, np.kron(Dp, IB))
      + np.kron(Pe, np.kron(Dm, IB))
    )


def ECD_B(beta):
    """Echoed Conditional Displacement on mode B, conditioned on the qubit."""
    Dp = displacement( beta / 2, "B")
    Dm = displacement(-beta / 2, "B")
    return (
        np.kron(Pg, np.kron(IA, Dp))
      + np.kron(Pe, np.kron(IA, Dm))
    )


def embedded_rotation(theta, phi):
    """Qubit rotation lifted to the full qubit (x) A (x) B Hilbert space."""
    R = Rphi(theta, phi)
    return np.kron(R, np.eye(cutoffA * cutoffB))


## 4. Target state: two-mode squeezed vacuum

The target unitary $S_{AB}(r) = \exp[r(a_A^\dagger a_B^\dagger - a_A a_B)]$
acts only on the cavities. Applied to $|0,0\rangle$ it produces the two-mode
squeezed vacuum,

$$
|\mathrm{TMSV}(r)\rangle
  = \frac{1}{\cosh r}\sum_{n=0}^{\infty}(\tanh r)^{n}\,|n,n\rangle_{A,B},
$$

whose photon-number distribution is thermal in each mode with mean occupation
$\sinh^{2}(r)$. For $r=0.5$ this is $\sinh^{2}(0.5)\approx 0.272$, so a cutoff
of ~10 on each mode is more than enough.


In [ ]:
def two_mode_squeezing(r):
    """Exact two-mode squeezing unitary on the truncated A (x) B space."""
    a1 = np.kron(aA, IB)
    a2 = np.kron(IA, aB)
    ad1, ad2 = a1.conj().T, a2.conj().T
    G = ad1 @ ad2 - a1 @ a2
    return expm(r * G)


# ---- Build the initial and target states ---------------------------------
vacA = np.zeros(cutoffA); vacA[0] = 1
vacB = np.zeros(cutoffB); vacB[0] = 1
vac2 = np.kron(vacA, vacB)          # |0,0>_{A,B}

g    = np.array([1, 0])             # |g> on the ancilla
psi0 = np.kron(g, vac2)             # |g> (x) |0,0>: full initial state

Utms         = two_mode_squeezing(r_target)
target_state = Utms @ vac2          # target lives on the cavities only


## 5. Variational ansatz and cost function

Each layer contributes 6 real parameters:

* $\mathrm{Re}\,\beta_A,\ \mathrm{Im}\,\beta_A$ — ECD amplitude on mode $A$
* $\mathrm{Re}\,\beta_B,\ \mathrm{Im}\,\beta_B$ — ECD amplitude on mode $B$
* $\theta,\ \varphi$ — qubit rotation angle and axis

The total unitary is $U(\vec p) = \prod_{k=1}^{N_{\text{layers}}}
R^{(k)}\,\mathrm{ECD}_B^{(k)}\,\mathrm{ECD}_A^{(k)}$.
After acting on $|g\rangle\otimes|0,0\rangle$, the ancilla is projected onto
$|g\rangle$ (heralding). The success-conditioned fidelity is
$F(\vec p) = |\langle\mathrm{TMSV}(r)|\Pi_g U(\vec p)|g,0,0\rangle|^{2}/
\|\Pi_g U(\vec p)|g,0,0\rangle\|^{2}$, and the optimizer minimizes $1-F$.

Note that this cost is *normalized* by the heralding probability, so a very
low-probability, high-fidelity outcome can dominate the landscape.


In [ ]:
def ansatz(params):
    """Build the full ECD+R unitary from a flat parameter vector."""
    U = np.eye(2 * cutoffA * cutoffB, dtype=complex)

    idx = 0
    for _ in range(Nlayers):
        # Complex ECD amplitudes packed as (Re, Im) pairs
        betaA = params[idx]     + 1j * params[idx + 1]; idx += 2
        betaB = params[idx]     + 1j * params[idx + 1]; idx += 2
        # Qubit rotation angle and axis
        theta = params[idx]
        phi   = params[idx + 1]; idx += 2

        # Apply in order:  ECD_A  ->  ECD_B  ->  R(theta, phi)
        U = ECD_A(betaA)             @ U
        U = ECD_B(betaB)             @ U
        U = embedded_rotation(theta, phi) @ U

    return U


def project_g(state):
    """Project the joint qubit-cavity state onto the ancilla |g> block."""
    dim = cutoffA * cutoffB
    return state[:dim]      # first `dim` amplitudes correspond to qubit=|g>


def fidelity(params):
    """Heralded fidelity against the target two-mode squeezed vacuum."""
    U   = ansatz(params)
    psi = project_g(U @ psi0)

    norm = np.linalg.norm(psi)
    if norm < 1e-12:
        return 0.0                     # heralding failed numerically

    psi /= norm                        # renormalize the heralded branch
    overlap = np.vdot(target_state, psi)
    return np.abs(overlap) ** 2


def cost(params):
    """Infidelity, the quantity the optimizer minimizes."""
    return 1.0 - fidelity(params)


# ---- Simple stateful callback for progress printouts ---------------------
_history = []
def callback(xk):
    _history.append(fidelity(xk))
    if len(_history) % 10 == 0:
        print(f"Step {len(_history):4d}   Fidelity = {_history[-1]:.8f}")


## 6. Optimization

The parameter landscape of ECD ansaetze is famously non-convex and littered
with local minima, and BFGS with a single random initialization is a fairly
weak solver here. For a serious study one should:

* run several restarts (e.g. 20–100) and keep the best;
* use layer-by-layer warm starts (add a new near-identity layer once the
  previous depth has converged);
* switch to a gradient method that exploits analytic gradients (autograd /
  JAX) for large cutoffs.

The cell below does a single BFGS run to keep the example lightweight.


In [ ]:
# 6 real parameters per layer (Re/Im of two betas, plus theta and phi)
nparams = Nlayers * 6

# Small random init; the ansatz is near the identity at params ~ 0
rng = np.random.default_rng(seed=0)
x0  = 0.1 * rng.standard_normal(nparams)

result = minimize(
    cost,
    x0,
    method="BFGS",
    callback=callback,
    options={"maxiter": 300, "disp": True},
)

print()
print("Optimization finished")
print(f"Final fidelity = {fidelity(result.x):.6f}")
print()

# Pretty-print the recovered gate parameters
idx = 0
for k in range(Nlayers):
    betaA = result.x[idx]     + 1j * result.x[idx + 1]; idx += 2
    betaB = result.x[idx]     + 1j * result.x[idx + 1]; idx += 2
    theta = result.x[idx]
    phi   = result.x[idx + 1]; idx += 2

    print(f"Layer {k}")
    print(f"  betaA = {betaA}")
    print(f"  betaB = {betaB}")
    print(f"  theta = {theta}")
    print(f"  phi   = {phi}")
    print()


## 7. Discussion

**What we saw.** With a single layer, `Nlayers = 1`, the optimizer plateaus at
a fidelity of order $0.7$–$0.8$ against $|\mathrm{TMSV}(0.5)\rangle$. This is
expected: one ECD on each mode plus a single qubit rotation cannot generate
the required $a_A^\dagger a_B^\dagger$–type correlations except through the
qubit as an intermediary, and one round of qubit entanglement is simply not
enough to build up meaningful two-mode entanglement.

**Why depth helps.** Two-mode squeezing is a Gaussian, but non-local, gate.
Each ECD layer entangles the qubit with one cavity; a subsequent qubit
rotation then converts that entanglement into a cavity–cavity correlation
after the next ECD. Roughly, ECD+R depth $N_{\text{layers}}$ enables a
Trotterized expansion of the joint Hamiltonian $a_A^\dagger a_B^\dagger -
\text{h.c.}$; empirically, fidelity climbs quickly with the first few extra
layers and then saturates near unity once the ansatz has enough freedom to
reproduce the required Bogoliubov transformation on the truncated basis.

**Truncation pitfalls.** `cutoffB = 2` in the original code is dangerous even
at $r=0.5$: it only allows $|0\rangle_B$ and $|1\rangle_B$, whereas
$|\mathrm{TMSV}(r)\rangle$ has support on all diagonal Fock states
$|n,n\rangle$. Raising both cutoffs to at least $\lceil 3\sinh^{2}(r) + 5
\rceil$ removes any truncation-induced fidelity ceiling.

**Heralding.** Because the fidelity is normalized by the projection onto
$|g\rangle$, the optimizer can sometimes prefer solutions with a small
success probability. For experimentally realistic protocols one should also
track $\|\Pi_g U|\psi_0\rangle\|^{2}$ and either add it to the cost as a
soft penalty (e.g. $1 - F + \lambda(1 - P_g)$) or require the ancilla to be
disentangled at the end (deterministic protocol).

**Practical suggestions.**

* Sweep `Nlayers` from 1 to, say, 6 and warm-start each depth from the
  previous optimum with the new layer initialized close to identity.
* Use several random restarts per depth; the ECD landscape is riddled with
  local minima.
* Track heralding probability alongside fidelity.
* Once satisfied, verify the recovered angles on a more physical simulator
  (e.g. `qutip` or an ECD-toolkit-style master equation) that includes
  finite-time distortion and ancilla decay, so the parameters transfer to a
  real experiment.
